                # 03 Model Training Lab

                Purpose: pull latest code, run baseline models, train tabular
                Phase 1 models, write model cards and summary predictions, and
                generate calibration diagnostics. This is not a backtest.
                


## 1. Mount Drive And Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)


## 2. Pull Latest Code


In [ ]:
import subprocess
from datetime import datetime, timezone
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
ALLOW_COLAB_RESET_TO_REMOTE = True
project_path = Path(PROJECT_ROOT)


def run_command(command, *, cwd=None, check=True):
    proc = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout.strip())
    if proc.stderr:
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(
            "Command failed with exit code "
            f"{proc.returncode}: {' '.join(command)}"
        )
    return proc


def git(args, *, check=True):
    return run_command(["git", "-C", PROJECT_ROOT, *args], check=check)


sync_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

if (project_path / ".git").exists():
    git(["remote", "set-url", "origin", GITHUB_REPO_URL])
    git(["fetch", "origin", GITHUB_BRANCH])

    tracked_status = git(["status", "--short", "--untracked-files=no"], check=False).stdout.strip()
    if tracked_status:
        print("Tracked local changes detected. Saving them to git stash before sync:")
        print(tracked_status)
        git(["stash", "push", "-m", f"colab-auto-stash-before-sync-{sync_id}"])

    current_branch = git(["rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
    local_main_exists = git(["show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"], check=False).returncode == 0

    if current_branch != GITHUB_BRANCH:
        if local_main_exists:
            git(["checkout", GITHUB_BRANCH])
        else:
            git(["checkout", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"])

    can_fast_forward = git(["merge-base", "--is-ancestor", "HEAD", f"origin/{GITHUB_BRANCH}"], check=False).returncode == 0
    if can_fast_forward:
        git(["merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
    elif ALLOW_COLAB_RESET_TO_REMOTE:
        backup_branch = f"colab-backup-before-sync-{sync_id}"
        git(["branch", backup_branch, "HEAD"], check=False)
        git(["reset", "--hard", f"origin/{GITHUB_BRANCH}"])
        print(
            "Local checkout was not fast-forwardable. "
            f"Previous HEAD was preserved as branch: {backup_branch}. "
            f"Working tree now matches origin/{GITHUB_BRANCH}."
        )
    else:
        raise RuntimeError(
            "Local checkout is not fast-forwardable. Set "
            "ALLOW_COLAB_RESET_TO_REMOTE = True to preserve the old HEAD in a "
            "backup branch and sync to origin/main."
        )
else:
    project_path.mkdir(parents=True, exist_ok=True)
    visible_items = [p.name for p in project_path.iterdir() if not p.name.startswith(".")]
    if visible_items:
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a git checkout. "
            "Move existing Drive files aside or start with an empty project folder before cloning. "
            f"Visible items: {visible_items[:10]}"
        )
    run_command(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT])

if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Repository is synced and ready at:", PROJECT_ROOT)


## 3. Install Dependencies And Stage Helper


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"

def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Review Model Configuration


In [ ]:
                from pathlib import Path
                print((Path(PROJECT_ROOT) / "configs" / "models.yaml").read_text())
                


## 5. Run Baseline Models


In [ ]:
baseline_result = run_stage_checked("train_baselines", config_name="models")


## 6. Train Tabular ML Models


In [ ]:
tabular_result = run_stage_checked("train_tabular", config_name="models")


## 7. Run Calibration Diagnostics


In [ ]:
calibration_result = run_stage_checked("calibrate_models", config_name="models")


## 8. Inspect Leaderboards And Calibration


In [ ]:
                from pathlib import Path
                import pandas as pd

                experiment_dir = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID
                print("\nBaseline comparison")
                print(pd.read_csv(experiment_dir / "baseline_comparison.csv"))
                print("\nModel leaderboard")
                print(pd.read_csv(experiment_dir / "model_leaderboard.csv"))
                print("\nCalibration metrics")
                print(pd.read_csv(experiment_dir / "calibration_metrics.csv"))
                


## 9. Confirm Phase Boundary


In [ ]:
                print("Training complete for Phase 1 diagnostics.")
                print("No equity curve, drawdown, position sizing, stop-loss, take-profit, or trade journal was produced.")
                


In [ ]:
# Auto-close the Colab runtime after every previous cell has completed.
# Set this to False before running if you want to keep the session alive for inspection.
AUTO_CLOSE_COLAB_RUNTIME = True

if AUTO_CLOSE_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("All notebook cells completed. Releasing the Colab runtime now.")
        runtime.unassign()
    except Exception as exc:
        print(f"Auto-close skipped outside Colab or because runtime release failed: {exc}")
